# Fast-SAM3D on Kaggle — staged 16 GB T4 mode

This uses [wlfeng0509/Fast-SAM3D](https://github.com/wlfeng0509/Fast-SAM3D), an accelerator built on Meta's released SAM 3D Objects weights. It is **not** the unrelated medical-segmentation repository named FastSAM3D.

Before running: enable Internet and GPU, request access to `facebook/sam-3d-objects`, then attach a Kaggle secret named `HF_TOKEN`. A secret attachment must be made in the Kaggle UI after any API/CLI notebook push; do not paste the token into this notebook.

> Kaggle's two 16 GB T4s are separate memories. This notebook stages models through one T4 and exports a Gaussian-splat PLY. It deliberately does not download or run the mesh decoder.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
print([(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1)) for i in range(torch.cuda.device_count())])

WORKDIR = Path('/kaggle/working')
DATASET_CHECKPOINT_DIR = Path('/kaggle/input/sam3d-fast-checkpoints-private')
CHECKPOINT_NAMES = ['ss_generator.ckpt', 'ss_decoder.ckpt', 'slat_generator.ckpt', 'slat_decoder_gs.ckpt']
# Do this before any slow package installation. API-pushed Kaggle versions do
# not retain secret attachments, so this error is actionable and inexpensive.
if all((DATASET_CHECKPOINT_DIR / name).is_file() for name in CHECKPOINT_NAMES):
    HF_TOKEN = None
    print('Using private SAM checkpoint dataset; no Hugging Face secret is needed.')
else:
    try:
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError(
            'Attach a Kaggle secret named HF_TOKEN in the notebook UI, or attach the private sam3d-fast-checkpoints-private dataset.'
        ) from exc
    assert HF_TOKEN, 'The HF_TOKEN Kaggle secret is empty.'

REPO_DIR = WORKDIR / 'Fast-SAM3D'
PYTHON = Path(sys.executable)
SETUP_MARKER = WORKDIR / '.fastsam3d-kaggle-env-ready'
print('Secret and GPU preflight passed.')


In [ ]:
# Clone the actual image-to-3D Fast-SAM3D project. This is intentionally not
# arcadelab/FastSAM3D, which is a medical volumetric-segmentation project.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/wlfeng0509/Fast-SAM3D.git', str(REPO_DIR)], check=True)
print('Fast-SAM3D commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
# Fast-SAM3D's current source imports an optional Euler_easy_ss solver that is
# not defined anywhere in its solver module. The staged accelerated path below
# uses euler_faster_slat instead, so remove only the broken unused import/map.
flow_model = REPO_DIR / 'sam3d_objects' / 'model' / 'backbone' / 'generator' / 'flow_matching' / 'model.py'
solver = flow_model.with_name('solver.py')
if 'class Euler_easy_ss' not in solver.read_text() and 'Euler_easy_ss,' in flow_model.read_text():
    flow_source = flow_model.read_text().replace('    Euler_easy_ss,\n', '').replace('        \"euler_easy_ss\":Euler_easy_ss,\n', '')
    flow_model.write_text(flow_source)
    print('Patched Fast-SAM3D optional undefined Euler_easy_ss import.')
# Fast-SAM3D also defaults to a DINO repository path on the author's machine.
# Keep the same local-source behavior, but provide that repository on Kaggle.
DINO_REPO = REPO_DIR / 'third_party' / 'dinov2'
if not DINO_REPO.exists():
    DINO_REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/dinov2.git', str(DINO_REPO)], check=True)
dino_module = REPO_DIR / 'sam3d_objects' / 'model' / 'backbone' / 'dit' / 'embedder' / 'dino.py'
dino_source = dino_module.read_text()
author_dino_path = '/data3/wmq/Fast-sam3d-objects/checkpoints/torch-cache/hub/facebookresearch_dinov2_main'
if author_dino_path in dino_source:
    dino_module.write_text(dino_source.replace(author_dino_path, str(DINO_REPO)))
    print('Patched Fast-SAM3D DINO source path for Kaggle.')
assert (DINO_REPO / 'hubconf.py').is_file(), 'DINO clone is incomplete.'
# The repository computes FFT confidence scores on CPU, then combines them with
# CUDA sparse coordinates. Make the score tensor follow the coordinates' device.
inference_utils = REPO_DIR / 'sam3d_objects' / 'pipeline' / 'inference_utils.py'
utils_source = inference_utils.read_text()
old_scores_line = '    raw_scores = coords_scores[:, 0].float()'
new_scores_line = '    raw_scores = coords_scores[:, 0].to(device=device, dtype=torch.float32)'
if old_scores_line in utils_source:
    inference_utils.write_text(utils_source.replace(old_scores_line, new_scores_line))
    print('Patched Fast-SAM3D CPU/CUDA score-device mismatch.')

# This is the same CUDA/Python closure that passed the SAM 3D import preflight
# in the prior notebook. Fast-SAM3D adds only local Python modules plus Plotly.
if not SETUP_MARKER.exists():
    env = os.environ.copy()
    env.update({
        'PIP_NO_CACHE_DIR': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'MAX_JOBS': '2',
        'CUMM_DISABLE_JIT': '1', 'SPCONV_DISABLE_JIT': '1', 'LIDRA_SKIP_INIT': 'true',
        'PIP_FIND_LINKS': 'https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html',
    })
    constraints = WORKDIR / 'fastsam3d-constraints.txt'
    constraints.write_text('numpy==1.26.4\n')
    def pip_install(*packages, extra=()):
        subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--no-cache-dir', '--constraint', str(constraints), *extra, *packages], env=env, check=True)
    def check_imports(code, label):
        subprocess.run([str(PYTHON), '-c', code], cwd=REPO_DIR, env=env, check=True)
        print(label)

    pip_install('torch==2.5.1', 'torchvision==0.20.1', extra=('--index-url', 'https://download.pytorch.org/whl/cu121'))
    pip_install(
        'numpy==1.26.4', 'astor==0.8.1', 'easydict', 'einops-exts', 'fvcore', 'huggingface-hub[cli]<1.0',
        'hydra-core==1.3.2', 'igraph==0.11.8', 'lightning==2.3.3', 'loguru==0.7.2',
        'matplotlib==3.9.2', 'omegaconf', 'open3d==0.19.0', 'optree==0.14.1',
        'opencv-python-headless==4.9.0.80', 'pillow', 'plotly==5.24.1', 'plyfile', 'pymeshfix==0.17.0',
        'pyvista==0.44.2', 'roma==1.5.1', 'safetensors', 'scipy', 'timm==0.9.16',
        'transformers==4.46.3', 'tqdm', 'trimesh', 'xatlas==0.0.9',
    )
    pip_install('kaolin==0.17.0', 'spconv-cu121==2.3.8')
    check_imports("import numpy; assert numpy.__version__ == '1.26.4'; import spconv", 'NumPy/spconv ABI preflight passed.')
    pip_install('git+https://github.com/facebookresearch/pytorch3d.git@75ebeeaea0908c5527e7b1e305fbc7681382db47')
    pip_install('moderngl')
    pip_install('git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900', extra=('--no-deps',))
    pip_install('MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b', extra=('--no-deps',))
    pip_install('gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@2323de5905d5e90e035f792fe65bad0fedd413e7')
    subprocess.run([str(PYTHON), str(REPO_DIR / 'patching' / 'hydra')], cwd=REPO_DIR, env=env, check=True)
    env.update({'ATTN_BACKEND': 'sdpa', 'SPARSE_ATTN_BACKEND': 'sdpa'})
    check_imports(
        "import importlib, numpy, torch; import astor, igraph, kaolin, matplotlib, open3d, optree, plotly, pymeshfix, pyvista, spconv, timm, transformers, pytorch3d, gsplat; assert transformers.__version__ == '4.46.3'; modules=('fft.fft2d','fft.fft3d','faster_utils_ss','faster_utils_slat','token_slat.selection','moge.model.v1','sam3d_objects.config.utils','sam3d_objects.model.backbone.generator.flow_matching.model','sam3d_objects.model.backbone.generator.shortcut.model','sam3d_objects.model.backbone.tdfy_dit.models.structured_latent_vae.decoder_gs','sam3d_objects.pipeline.depth_models.moge','sam3d_objects.pipeline.inference_pipeline','sam3d_objects.pipeline.inference_pipeline_pointmap'); [importlib.import_module(m) for m in modules]; print(torch.__version__, torch.version.cuda, numpy.__version__)",
        'Fast-SAM3D runtime import preflight passed.'
    )
    SETUP_MARKER.touch()
else:
    print('Reusing the completed environment in this Kaggle session.')
print('Environment ready.')


In [ ]:
# Fetch only weights used by staged Gaussian output. The Fast-SAM3D repository
# already contains ss_generator_faster.yaml and slat_generator_faster.yaml.
import shutil
from huggingface_hub import snapshot_download

REPO_CHECKPOINT_DIR = REPO_DIR / 'checkpoints' / 'hf'
DOWNLOAD_DIR = REPO_DIR / 'checkpoints' / 'hf-download'
checkpoint_names = ['ss_generator.ckpt', 'ss_decoder.ckpt', 'slat_generator.ckpt', 'slat_decoder_gs.ckpt']
if all((DATASET_CHECKPOINT_DIR / name).is_file() for name in checkpoint_names):
    CHECKPOINT_DIR = DATASET_CHECKPOINT_DIR
    print('Using reusable private Kaggle checkpoint dataset:', CHECKPOINT_DIR)
else:
    CHECKPOINT_DIR = REPO_CHECKPOINT_DIR
    missing = [name for name in checkpoint_names if not (CHECKPOINT_DIR / name).is_file()]
if CHECKPOINT_DIR == REPO_CHECKPOINT_DIR and missing:
    free_gb = shutil.disk_usage(WORKDIR).free / 2**30
    # The four staged weights total 11.9 GB; leave room for public MoGe weights,
    # temporary Hub metadata, and the PLY without rejecting a normal Kaggle disk.
    assert free_gb >= 16, f'Need at least 16 GB free; only {free_gb:.1f} GB is available.'
    print('Downloading:', missing)
    snapshot_download(
        'facebook/sam-3d-objects', repo_type='model', token=HF_TOKEN, max_workers=1,
        allow_patterns=[f'checkpoints/{name}' for name in checkpoint_names],
        local_dir=str(DOWNLOAD_DIR),
    )
    # Keep an interrupted download directory intact until every file has finished,
    # so rerunning this cell resumes rather than starts the large files over.
    for name in checkpoint_names:
        source = DOWNLOAD_DIR / 'checkpoints' / name
        assert source.is_file(), f'Missing completed download: {source}'
        source.replace(CHECKPOINT_DIR / name)
    shutil.rmtree(DOWNLOAD_DIR)
elif CHECKPOINT_DIR == REPO_CHECKPOINT_DIR:
    print('Reusing all SAM 3D checkpoints.')
assert (REPO_CHECKPOINT_DIR / 'ss_generator_faster.yaml').is_file()
assert (REPO_CHECKPOINT_DIR / 'slat_generator_faster.yaml').is_file()
print('Checkpoints ready:', CHECKPOINT_DIR)


In [ ]:
# Replace these two paths with an attached Kaggle Dataset for your own object.
# The mask must match the image dimensions: white is the object, black is background.
EXAMPLE_DIR = REPO_DIR / 'notebook' / 'images' / 'shutterstock_stylish_kidsroom_1640806567'
INPUT_IMAGE = EXAMPLE_DIR / 'image.png'
INPUT_MASK = EXAMPLE_DIR / '14.png'
OUTPUT_DIR = WORKDIR / 'fastsam3d-output'

MAX_INPUT_SIDE = 768       # keep first 16 GB T4 test conservative
STAGE1_STEPS = 12          # Fast sparse-structure steps
STAGE2_STEPS = 12          # Fast SLaT steps
SEED = 42

assert INPUT_IMAGE.is_file(), f'Missing image: {INPUT_IMAGE}'
assert INPUT_MASK.is_file(), f'Missing mask: {INPUT_MASK}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Execute Fast-SAM3D in explicit GPU stages. Its public demo puts every model
# on one GPU simultaneously; that is not viable on a 16 GB T4.
RUNNER_SOURCE = r'''
import gc
import math
import os
from pathlib import Path

import numpy as np
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
from PIL import Image

os.environ.update({
    'ATTN_BACKEND': 'sdpa', 'SPARSE_ATTN_BACKEND': 'sdpa',
    'CUMM_DISABLE_JIT': '1', 'SPCONV_DISABLE_JIT': '1', 'LIDRA_SKIP_INIT': 'true',
})
from sam3d_objects.pipeline.inference_pipeline import InferencePipeline

repo = Path(os.environ['FASTSAM3D_REPO_DIR'])
checkpoint_dir = Path(os.environ['FASTSAM3D_CHECKPOINT_DIR'])
# DINO weights are fetched by torch.hub on first use. Keep them in the Kaggle
# working directory rather than Fast-SAM3D's author-specific cache path.
torch_cache_dir = repo / 'checkpoints' / 'torch-cache'
torch_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(torch_cache_dir)
image_path = Path(os.environ['FASTSAM3D_INPUT_IMAGE'])
mask_path = Path(os.environ['FASTSAM3D_INPUT_MASK'])
output_dir = Path(os.environ['FASTSAM3D_OUTPUT_DIR'])
max_input_side = int(os.environ['FASTSAM3D_MAX_INPUT_SIDE'])
stage1_steps = int(os.environ['FASTSAM3D_STAGE1_STEPS'])
stage2_steps = int(os.environ['FASTSAM3D_STAGE2_STEPS'])
seed = int(os.environ['FASTSAM3D_SEED'])
output_dir.mkdir(parents=True, exist_ok=True)
gpu, cpu = torch.device('cuda:0'), torch.device('cpu')

def clear_cuda(label):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'{label}: allocated={torch.cuda.memory_allocated() / 2**30:.2f} GB, reserved={torch.cuda.memory_reserved() / 2**30:.2f} GB')

def move_models(names, device):
    for name in names:
        model = pipeline.models[name]
        if model is not None:
            model.to(device)
    conditioners = {'ss_generator': 'ss_condition_embedder', 'slat_generator': 'slat_condition_embedder'}
    for name in names:
        conditioner = pipeline.condition_embedders.get(conditioners.get(name, ''))
        if conditioner is not None:
            conditioner.to(device)

def move_depth(device):
    pipeline.depth_model.model.to(device)
    pipeline.depth_model.device = device

# Upstream Fast-SAM3D always initializes a mesh decoder and its Gaussian-only
# decoder logs an unset mesh timer. Both are corrected locally, without changing
# Fast-SAM3D's accelerated sparse-structure or SLaT generators.
InferencePipeline.init_slat_decoder_mesh = lambda self, *args, **kwargs: None
def gaussian_only_decode(self, map_tokens, slat, formats):
    assert formats == ['gaussian'], formats
    with torch.no_grad():
        return {'gaussian': self.models['slat_decoder_gs'](slat)}
InferencePipeline.decode_slat = gaussian_only_decode

# Fast-SAM3D configuration files live in the cloned source; checkpoint weights
# may instead be mounted read-only from the private Kaggle dataset.
config_path = repo / 'checkpoints' / 'hf' / 'pipeline.yaml'
config = OmegaConf.load(config_path)
config.workspace_dir = str(checkpoint_dir)
config.device = 'cpu'
config.depth_model.device = 'cpu'
# The Fast-SAM3D repository ships a developer-local path that cannot exist on Kaggle.
config.depth_model.model.pretrained_model_name_or_path = 'Ruicheng/moge-vitl'
config.compile_model = False
config.decode_formats = ['gaussian']
config.slat_decoder_gs_4_config_path = None
config.slat_decoder_gs_4_ckpt_path = None
config.slat_decoder_mesh_config_path = None
config.slat_decoder_mesh_ckpt_path = None
config.ss_decoder_config_path = str(repo / 'checkpoints' / 'hf' / 'ss_decoder.yaml')
config.slat_decoder_gs_config_path = str(repo / 'checkpoints' / 'hf' / 'slat_decoder_gs.yaml')
config.ss_generator_config_path = str(repo / 'checkpoints' / 'hf' / 'ss_generator_faster.yaml')
config.slat_generator_config_path = str(repo / 'checkpoints' / 'hf' / 'slat_generator_faster.yaml')
pipeline = instantiate(config)
pipeline.device = gpu
pipeline.ss_params = {'ss_faster_stride': 3, 'ss_warmup': 2, 'ss_order': 1, 'ss_momentum_beta': 0.5}
pipeline.slat_params = {'slat_thresh': 1.5, 'slat_warmup': 3, 'slat_token_ratio': 0.1}
pipeline.mesh_params = {}
pipeline.enable_mesh = False
pipeline.hfer_2d = 0.0
# Keep Fast-SAM3D's token-pruning schedule consistent with the reduced T4 step count.
token_args = pipeline.models['slat_generator'].args
token_args.effective_steps = stage2_steps
token_args.full_sampling_end_steps = math.ceil(stage2_steps * 0.75)
token_args.anchor_step = max(1, math.floor(stage2_steps * 0.2))
print('Loaded Fast-SAM3D weights on CPU; accelerated generators are enabled.')

image = np.asarray(Image.open(image_path).convert('RGB'))
mask = np.asarray(Image.open(mask_path).convert('L')) > 127
assert image.shape[:2] == mask.shape[:2], 'Image and mask dimensions must match.'
height, width = image.shape[:2]
if max(height, width) > max_input_side:
    scale = max_input_side / max(height, width)
    size = (round(width * scale), round(height * scale))
    image = np.asarray(Image.fromarray(image).resize(size, Image.Resampling.LANCZOS))
    mask = np.asarray(Image.fromarray(mask.astype(np.uint8) * 255).resize(size, Image.Resampling.NEAREST)) > 127
    print(f'Resized input to {image.shape[1]}x{image.shape[0]}.')
rgba = np.concatenate([image, (mask.astype(np.uint8) * 255)[..., None]], axis=-1)

move_depth(gpu)
clear_cuda('MoGe loaded')
with torch.no_grad():
    pointmap = pipeline.compute_pointmap(rgba)['pointmap']
move_depth(cpu)
clear_cuda('MoGe offloaded')

ss_input = pipeline.preprocess_image(rgba, pipeline.ss_preprocessor, pointmap=pointmap)
del pointmap
move_models(('ss_generator', 'ss_decoder'), gpu)
clear_cuda('Fast sparse structure loaded')
torch.manual_seed(seed)
ss_return, map_tokens, coords_scores = pipeline.sample_sparse_structure(ss_input, inference_steps=stage1_steps)
ss_return.update(pipeline.pose_decoder(ss_return, scene_scale=ss_input.get('pointmap_scale'), scene_shift=ss_input.get('pointmap_shift')))
ss_return['scale'] = ss_return['scale'] * ss_return['downsample_factor']
coords = ss_return['coords']
move_models(('ss_generator', 'ss_decoder'), cpu)
del ss_input, ss_return
clear_cuda('Fast sparse structure offloaded')

slat_input = pipeline.preprocess_image(rgba, pipeline.slat_preprocessor)
move_models(('slat_generator',), gpu)
clear_cuda('Fast SLaT loaded')
slat = pipeline.sample_slat(slat_input, coords, inference_steps=stage2_steps, map_tokens=map_tokens, coords_scores=coords_scores)
move_models(('slat_generator',), cpu)
del slat_input, coords, map_tokens, coords_scores
clear_cuda('Fast SLaT offloaded')

move_models(('slat_decoder_gs',), gpu)
clear_cuda('Gaussian decoder loaded')
gaussian = pipeline.decode_slat(None, slat, ['gaussian'])['gaussian'][0]
ply_path = output_dir / 'fastsam3d_reconstruction.ply'
gaussian.save_ply(ply_path)
print('Saved:', ply_path)
'''
compile(RUNNER_SOURCE, 'run_fastsam3d_staged.py', 'exec')
runner = REPO_DIR / 'run_fastsam3d_staged.py'
runner.write_text(RUNNER_SOURCE)

runner_env = os.environ.copy()
runner_env.update({
    'CUDA_VISIBLE_DEVICES': '0', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'FASTSAM3D_REPO_DIR': str(REPO_DIR), 'FASTSAM3D_CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'FASTSAM3D_INPUT_IMAGE': str(INPUT_IMAGE), 'FASTSAM3D_INPUT_MASK': str(INPUT_MASK),
    'FASTSAM3D_OUTPUT_DIR': str(OUTPUT_DIR), 'FASTSAM3D_MAX_INPUT_SIDE': str(MAX_INPUT_SIDE),
    'FASTSAM3D_STAGE1_STEPS': str(STAGE1_STEPS), 'FASTSAM3D_STAGE2_STEPS': str(STAGE2_STEPS),
    'FASTSAM3D_SEED': str(SEED), 'ATTN_BACKEND': 'sdpa', 'SPARSE_ATTN_BACKEND': 'sdpa',
    'TORCH_HOME': str(REPO_DIR / 'checkpoints' / 'torch-cache'),
    'CUMM_DISABLE_JIT': '1', 'SPCONV_DISABLE_JIT': '1', 'LIDRA_SKIP_INIT': 'true',
})
subprocess.run([str(PYTHON), str(runner)], cwd=REPO_DIR, env=runner_env, check=True)

# Only remove weights after a successful PLY write. A failed run keeps them for
# diagnosis/retry in the same Kaggle session.
if CHECKPOINT_DIR == REPO_DIR / 'checkpoints' / 'hf':
    for name in checkpoint_names:
        (CHECKPOINT_DIR / name).unlink(missing_ok=True)
print('Output retained in', OUTPUT_DIR)


## Unity hand-off

Download `fastsam3d-output/fastsam3d_reconstruction.ply` from the Kaggle output. It is a Gaussian-splat point-cloud reconstruction, not a textured mesh. Keep it as the portal-reveal asset or convert it to the rendering format used by your Unity splat renderer.
